# RAG from Scratch to Production — Episode 4/20
## Chunking: cutting text without cutting answers in half

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/04_chunking.ipynb)

Episode 3 turned files into clean text. Now we cut that text into **chunks**, the small pieces that retrieval searches and the LLM reads.

Every answer the bot can give lives inside one chunk. Cut in the wrong place, and the answer is split across two chunks that each make no sense alone.

**In this episode:**
1. Build a **known-facts test**: 8 questions whose answers we know are in the document.
2. Try the obvious approach, **fixed-size chunks**, and watch it cut words and facts in half.
3. Build the **recursive splitter** from scratch, and prove it gives exactly the same chunks as LangChain's, which the [production app](https://github.com/JakkulaMahendhar/rag-chatbot) uses.
4. Measure the 3 knobs: **chunk size**, **overlap**, and **section titles**.

## 0. Setup

LangChain's splitter is only installed to check that our version matches it. The LLM is only used for a few before/after answers. Same `generate()` as earlier episodes: Gemini (free key, add it as a Colab secret named `GEMINI_API_KEY`) or local Ollama.

In [ ]:
%pip install -q langchain-text-splitters google-generativeai requests

In [1]:
import os, re, requests

PROVIDER = os.getenv("LLM_PROVIDER", "gemini")   # "gemini" or "ollama"
GEMINI_MODEL = "gemini-3.6-flash"
OLLAMA_MODEL = "llama3.1"


def _gemini_key():
    try:
        from google.colab import userdata  # Colab secrets
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.getenv("GEMINI_API_KEY") or __import__("getpass").getpass("GEMINI_API_KEY: ")


if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=_gemini_key())
    _model = genai.GenerativeModel(GEMINI_MODEL)


def generate(prompt: str) -> str:
    """One interface, two providers. Temperature 0 so results are repeatable."""
    if PROVIDER == "gemini":
        return _model.generate_content(
            prompt, generation_config={"temperature": 0}
        ).text.strip()
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
              "options": {"temperature": 0}},
        timeout=300,
    )
    r.raise_for_status()
    return r.json()["response"].strip()


print(f"Using {PROVIDER}")

Using ollama


## 1. The document and the known-facts test

The NovaTech handbook again, as the clean text Episode 3's parser would produce: 6 sections separated by blank lines.

In [2]:
HANDBOOK = """NovaTech Solutions - Employee Handbook (2026 edition)

Remote Work
Employees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week. Managers may approve extra remote days during school holidays.

Annual Leave
Full-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year; the rest expire on 31 March. Leave requests of more than 10 consecutive days need approval at least one month in advance.

Expenses
Business expenses must be submitted within 30 days with a receipt. The meal allowance during business travel is capped at 40 USD per day. Expense claims above 500 USD must also be approved by the finance director before payment.

Learning Budget
Every employee has an annual learning budget of 1,200 USD for courses, certifications and conferences. It does not roll over: whatever is not spent by 31 December is lost. Requests are approved by your direct manager.

Equipment
Every employee receives a laptop, refreshed every 3 years. Lost or stolen devices must be reported to IT within 24 hours. Personal devices may only access company email after enrolling in device management.

Security
Multi-factor authentication is mandatory on all company accounts. Suspected security incidents must be reported to the security team within 1 hour. Passwords must be at least 14 characters long and must never be reused."""

print(f"{len(HANDBOOK):,} characters")

1,511 characters


How do we know if one way of chunking is better than another? **We test it.** Below are 8 questions, each with a short piece of text that the retrieved chunk **must** contain for the answer to be possible.

If retrieval returns a chunk without that text, no LLM can answer correctly. This check needs no LLM at all, so it's fast and free to run after every change.

In [3]:
KNOWN_FACTS = [
    ("Does the learning budget roll over?",                      "not roll over"),
    ("How many remote days per week are allowed?",               "2 days per week"),
    ("Who must approve expense claims above 500 USD?",           "finance director"),
    ("How many days of annual leave do full-time employees get?", "24 days"),
    ("How quickly must lost devices be reported?",               "24 hours"),
    ("How long must passwords be?",                              "14 characters"),
    ("When do unused leave days expire?",                        "31 March"),
    ("Who approves requests for the learning budget?",           "direct manager"),
]

STOPWORDS = {"a", "an", "the", "is", "are", "and", "or", "of", "to", "in", "on",
             "at", "for", "per", "what", "how", "many", "much", "does", "do", "i",
             "my", "can", "get", "it", "be", "must", "by", "with", "who", "when"}

def words(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS}


def retrieve(question: str, chunks: list[str]) -> str:
    """Episode 1's keyword retriever: the chunk sharing the most question words."""
    q = words(question)
    return max(chunks, key=lambda c: len(q & words(c)) / len(q))


def known_facts_test(chunks: list[str], label: str = "") -> int:
    hits = [fact in retrieve(q, chunks) for q, fact in KNOWN_FACTS]
    marks = " ".join("✅" if h else "❌" for h in hits)
    print(f"{label:<26} {len(chunks):>3} chunks   {sum(hits)}/{len(hits)}   {marks}")
    return sum(hits)

And the Episode 1 RAG step, so we can see what the user would get:

In [4]:
FALLBACK = "I don't have enough information from the provided documents."

def rag(question: str, chunks: list[str]) -> str:
    context = retrieve(question, chunks)
    print(f"Retrieved chunk: {context!r}")
    return generate(f"""Answer only using the context below. If the answer is not in the
context, respond exactly: "{FALLBACK}"

Context:
{context}

Question: {question}
Answer:""")

## 2. Strategy 1: fixed-size chunks

The obvious approach: every 300 characters, cut.

In [5]:
def split_fixed(text: str, size: int) -> list[str]:
    return [text[i:i + size] for i in range(0, len(text), size)]


fixed = split_fixed(HANDBOOK, 300)
for i, chunk in enumerate(fixed):
    print(f"chunk {i}: ...{chunk[-28:]!r}  |  next starts {fixed[i + 1][:12]!r}..." if i + 1 < len(fixed) else f"chunk {i}: ...{chunk[-28:]!r}")

chunk 0: ...'ra remote days during school'  |  next starts ' holidays.\n\n'...
chunk 1: ...'month in advance.\n\nExpenses\n'  |  next starts 'Business exp'...
chunk 2: ...'al learning budget of 1,200 '  |  next starts 'USD for cour'...
chunk 3: ...'d to IT within 24 hours. Per'  |  next starts 'sonal device'...
chunk 4: ...'aracters long and must never'  |  next starts ' be reused.'...
chunk 5: ...' be reused.'


Look at the cut points. The splitter has no idea what it is cutting:
- `"learning budget of 1,200 "` | `"USD for courses"`: a number separated from its unit.
- `"... within 24 hours. Per"` | `"sonal devices"`: a word cut in half.

Now the known-facts test:

In [6]:
known_facts_test(fixed, "fixed, 300 chars");

fixed, 300 chars             6 chunks   6/8   ❌ ✅ ✅ ✅ ✅ ✅ ✅ ❌


2 of 8 facts can't be found. Here's the first one from the user's side:

In [7]:
print(rag("Does the learning budget roll over?", fixed))

Retrieved chunk: 'Business expenses must be submitted within 30 days with a receipt. The meal allowance during business travel is capped at 40 USD per day. Expense claims above 500 USD must also be approved by the finance director before payment.\n\nLearning Budget\nEvery employee has an annual learning budget of 1,200 '


I don't have enough information from the provided documents.


The cut landed in the middle of the Learning Budget section. "Learning budget" is in one chunk and "It does not roll over" is in the next, so neither chunk can answer the question alone.

## 3. Strategy 2: the recursive splitter

A better rule: **cut at the most natural break that keeps chunks under the size limit.**

1. Try to split on **blank lines** (paragraphs and sections).
2. If a piece is still too big, split it on **line breaks**.
3. Still too big? Split on **spaces** (between words).
4. Last resort: split between characters.

Then glue small neighbouring pieces back together until adding the next one would go over the limit.

This is LangChain's `RecursiveCharacterTextSplitter`, which the [production app](https://github.com/JakkulaMahendhar/rag-chatbot) uses in `app/services/chunker.py`. Here it is from scratch:

In [8]:
def split_recursive(text: str, size: int, overlap: int = 0,
                    separators=("\n\n", "\n", " ", "")) -> list[str]:
    # Use the first separator that actually appears in the text.
    sep, finer = separators[-1], []
    for i, s in enumerate(separators):
        if s == "" or s in text:
            sep, finer = s, list(separators[i + 1:])
            break

    # Split on it, keeping the separator at the start of each piece.
    if sep:
        parts = text.split(sep)
        pieces = [parts[0]] + [sep + p for p in parts[1:]]
    else:
        pieces = list(text)

    chunks, small = [], []
    for piece in filter(None, pieces):
        if len(piece) < size:
            small.append(piece)            # small enough: merge it with its neighbours later
            continue
        if small:
            chunks += merge(small, size, overlap)
            small = []
        # Too big: split this piece again with a finer separator.
        chunks += split_recursive(piece, size, overlap, finer) if finer else [piece]
    if small:
        chunks += merge(small, size, overlap)
    return chunks


def merge(pieces: list[str], size: int, overlap: int) -> list[str]:
    """Glue pieces into chunks of up to `size` characters.
    Each new chunk starts with up to `overlap` characters from the end of the previous one."""
    chunks, window, total = [], [], 0
    for piece in pieces:
        if window and total + len(piece) > size:
            chunks.append("".join(window).strip())
            # Drop pieces from the front until only `overlap` characters remain.
            while total > overlap or (total + len(piece) > size and total > 0):
                total -= len(window.pop(0))
        window.append(piece)
        total += len(piece)
    chunks.append("".join(window).strip())
    return [c for c in chunks if c]

**Is it really the same as LangChain's?** Let's compare both on 6 different settings, including production's (1,000 characters, 200 overlap):

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

for size, overlap in [(1000, 200), (500, 100), (300, 0), (300, 60), (150, 30), (100, 0)]:
    langchain = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap, length_function=len
    ).split_text(HANDBOOK)
    ours = split_recursive(HANDBOOK, size, overlap)
    print(f"size={size:<5} overlap={overlap:<4} {len(ours):>2} chunks   identical: {ours == langchain}")

size=1000  overlap=200   2 chunks   identical: True
size=500   overlap=100   4 chunks   identical: True
size=300   overlap=0     7 chunks   identical: True
size=300   overlap=60    7 chunks   identical: True
size=150   overlap=30   19 chunks   identical: True
size=100   overlap=0    25 chunks   identical: True


Identical, every time. No magic inside the library: it's those ~40 lines.

Now the same 300-character limit, cut the recursive way:

In [10]:
recursive = split_recursive(HANDBOOK, 300)
for i, chunk in enumerate(recursive):
    print(f"chunk {i} ({len(chunk)} chars): {chunk[:60]!r}...")

print()
known_facts_test(fixed, "fixed, 300 chars")
known_facts_test(recursive, "recursive, 300 chars");

chunk 0 (53 chars): 'NovaTech Solutions - Employee Handbook (2026 edition)'...
chunk 1 (255 chars): 'Remote Work\nEmployees may work remotely up to 2 days per wee'...
chunk 2 (277 chars): 'Annual Leave\nFull-time employees receive 24 days of paid ann'...
chunk 3 (237 chars): 'Expenses\nBusiness expenses must be submitted within 30 days '...
chunk 4 (233 chars): 'Learning Budget\nEvery employee has an annual learning budget'...
chunk 5 (216 chars): 'Equipment\nEvery employee receives a laptop, refreshed every '...
chunk 6 (228 chars): 'Security\nMulti-factor authentication is mandatory on all com'...

fixed, 300 chars             6 chunks   6/8   ❌ ✅ ✅ ✅ ✅ ✅ ✅ ❌
recursive, 300 chars         7 chunks   8/8   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅


Every chunk is now one whole section, because each section fits under 300 characters. **8/8.**

In [11]:
print(rag("Does the learning budget roll over?", recursive))

Retrieved chunk: 'Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses, certifications and conferences. It does not roll over: whatever is not spent by 31 December is lost. Requests are approved by your direct manager.'


No, it does not roll over. Whatever is not spent by 31 December is lost.


## 4. Knob 1: chunk size

Bigger chunks keep more context together. But every retrieved chunk goes into the prompt, so bigger chunks also mean **more tokens per question** (cost and speed) and more irrelevant text around the answer.

Let's measure across sizes:

In [12]:
print(f"{'':<26} {'':>3}          {'facts found'}")
for size in [100, 150, 200, 300, 500, 1000, 2000]:
    chunks = split_recursive(HANDBOOK, size)
    known_facts_test(chunks, f"recursive, {size} chars")
    print(f"{'':<26} avg {sum(map(len, chunks)) // len(chunks):>4} characters sent to the LLM per question")

                                        facts found
recursive, 100 chars        25 chunks   2/8   ❌ ❌ ❌ ✅ ❌ ✅ ❌ ❌
                           avg   59 characters sent to the LLM per question
recursive, 150 chars        19 chunks   6/8   ✅ ✅ ✅ ✅ ✅ ✅ ❌ ❌
                           avg   78 characters sent to the LLM per question
recursive, 200 chars        19 chunks   6/8   ✅ ✅ ❌ ✅ ✅ ✅ ✅ ❌
                           avg   78 characters sent to the LLM per question
recursive, 300 chars         7 chunks   8/8   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅
                           avg  214 characters sent to the LLM per question
recursive, 500 chars         4 chunks   8/8   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅
                           avg  376 characters sent to the LLM per question
recursive, 1000 chars        2 chunks   8/8   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅
                           avg  754 characters sent to the LLM per question
recursive, 2000 chars        1 chunks   8/8   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅
                           avg 1511 characters sent to the

- **Too small (100–200):** sentences lose their subject. A chunk saying *"Requests are approved by your direct manager."* doesn't say requests *for what*.
- **300 and up:** each section fits in one chunk, so all 8 facts are found.
- **2,000:** the whole handbook is one chunk. 8/8 again, but every question now sends the **entire document** to the LLM. That's Episode 1's "just paste everything" problem.

There's no universal best size. It depends on your documents: short policy sections like these, long legal clauses, or code. **Measure it with your own known facts.**

## 5. Knob 2: overlap

Overlap repeats the end of each chunk at the start of the next, so a sentence cut at a boundary also appears whole in the next chunk. Production uses 200 characters of overlap on 1,000-character chunks.

Here's the Learning Budget section at 200 characters, with and without overlap:

In [13]:
for overlap in [0, 50]:
    print(f"── overlap = {overlap}")
    for chunk in split_recursive(HANDBOOK, 200, overlap):
        if "budget" in chunk or "manager" in chunk:
            print(f"   {chunk!r}")

── overlap = 0
   'Every employee has an annual learning budget of 1,200 USD for courses, certifications and conferences. It does not roll over: whatever is not spent by 31 December is lost. Requests are approved by'
   'your direct manager.'
── overlap = 50
   'Every employee has an annual learning budget of 1,200 USD for courses, certifications and conferences. It does not roll over: whatever is not spent by 31 December is lost. Requests are approved by'
   'by 31 December is lost. Requests are approved by your direct manager.'


- **No overlap:** the last chunk is just `'your direct manager.'`, an orphan with no meaning.
- **With overlap:** it becomes *"...Requests are approved by your direct manager."*, a complete sentence.

But it still doesn't say **learning budget**. Overlap repairs a cut sentence. It can't bring back a subject that was 2 sentences earlier. Does it change the test results?

In [14]:
for size in [100, 150, 200, 300]:
    known_facts_test(split_recursive(HANDBOOK, size, 0), f"{size} chars, no overlap")
    known_facts_test(split_recursive(HANDBOOK, size, size // 4), f"{size} chars, 25% overlap")

100 chars, no overlap       25 chunks   2/8   ❌ ❌ ❌ ✅ ❌ ✅ ❌ ❌
100 chars, 25% overlap      26 chunks   1/8   ❌ ❌ ❌ ✅ ❌ ❌ ❌ ❌
150 chars, no overlap       19 chunks   6/8   ✅ ✅ ✅ ✅ ✅ ✅ ❌ ❌
150 chars, 25% overlap      20 chunks   6/8   ✅ ✅ ✅ ✅ ✅ ✅ ❌ ❌
200 chars, no overlap       19 chunks   6/8   ✅ ✅ ❌ ✅ ✅ ✅ ✅ ❌
200 chars, 25% overlap      19 chunks   6/8   ✅ ✅ ❌ ✅ ✅ ✅ ✅ ❌
300 chars, no overlap        7 chunks   8/8   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅
300 chars, 25% overlap       7 chunks   8/8   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅


On this document, overlap didn't find **a single extra fact**. (At 100 characters it even lost one: the repeated text changed which chunk scored highest.)

That doesn't make overlap useless. It's cheap insurance for sentences cut at a boundary, and it matters more with longer chunks and with the embedding search coming in Episode 5. But it's not the fix for lost context.

## 6. Knob 3: put the section title on every chunk

The problem with small chunks is that they lose their **subject**. The subject is usually the section title. So write it into every chunk from that section:

In [15]:
def split_with_titles(text: str, size: int, overlap: int = 0) -> list[str]:
    chunks = []
    for section in text.split("\n\n")[1:]:              # skip the document title
        title, body = section.split("\n", 1)
        room = size - len(title) - 2                      # keep the whole chunk under `size`
        chunks += [f"{title}: {c}" for c in split_recursive(body, room, overlap)]
    return chunks


print([c for c in split_with_titles(HANDBOOK, 150) if "manager" in c])
print()
for size in [100, 150, 200, 300]:
    known_facts_test(split_recursive(HANDBOOK, size), f"{size} chars, plain")
    known_facts_test(split_with_titles(HANDBOOK, size), f"{size} chars, + titles")

['Learning Budget: whatever is not spent by 31 December is lost. Requests are approved by your direct manager.']

100 chars, plain            25 chunks   2/8   ❌ ❌ ❌ ✅ ❌ ✅ ❌ ❌
100 chars, + titles         19 chunks   5/8   ✅ ✅ ❌ ✅ ❌ ❌ ✅ ✅
150 chars, plain            19 chunks   6/8   ✅ ✅ ✅ ✅ ✅ ✅ ❌ ❌
150 chars, + titles         12 chunks   7/8   ✅ ✅ ✅ ✅ ✅ ✅ ❌ ✅
200 chars, plain            19 chunks   6/8   ✅ ✅ ❌ ✅ ✅ ✅ ✅ ❌
200 chars, + titles         12 chunks   6/8   ✅ ✅ ❌ ✅ ✅ ✅ ✅ ❌
300 chars, plain             7 chunks   8/8   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅
300 chars, + titles          6 chunks   8/8   ✅ ✅ ✅ ✅ ✅ ✅ ✅ ✅


The orphan sentence now reads *"Learning Budget: ...Requests are approved by your direct manager."*, and small chunks find up to **3 more facts** (at 100 characters).

The same question, from the user's side, at 150 characters:

In [16]:
question = "Who approves requests for the learning budget?"
print(rag(question, split_recursive(HANDBOOK, 150)))
print()
print(rag(question, split_with_titles(HANDBOOK, 150)))

Retrieved chunk: 'Learning Budget'


I don't have enough information from the provided documents.

Retrieved chunk: 'Learning Budget: whatever is not spent by 31 December is lost. Requests are approved by your direct manager.'


Your direct manager.


Without titles, the best-scoring chunk was the heading **"Learning Budget"** on its own: the right topic, zero content. With titles, the heading travels with the sentence that answers the question.

## 7. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) uses `RecursiveCharacterTextSplitter` with **1,000 characters and 200 overlap**, and gives every chunk an id like `"<document_id>-<index>"` so answers can cite their source (Episode 2's stage 9). On this 1,511-character handbook, that's just 2 chunks. On a 300-page PDF (about 750,000 characters), roughly 900.

It doesn't add section titles. Our test suggests that's worth trying, especially if chunks ever get smaller.

## Recap

| Approach | Facts found (300 chars) | Problem |
|---|---|---|
| Fixed-size | 6/8 | Cuts words, numbers and facts in half |
| Recursive (production) | **8/8** | Small chunks can still lose their subject |
| + overlap | No change here | Repairs cut sentences, not lost context |
| + section titles | Up to +3 facts on small chunks | Needs documents with clear headings |

**The rule:** don't guess your chunk settings. Write down questions you know the answers to, and **test**.

**Next, Episode 5:** keyword search still can't match "WFH" with "remote". **Embeddings** match on meaning instead of words.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)